# 00 — Tensor basics

Everything you need to write `hedging_gains` in `dhbench/pnl.py`. Nothing more.

**How this works:** learn one small thing, then immediately practise that one thing.
Seven rounds. Each practice cell has a `TODO` and tells you the answer to expect, so you
can check yourself without asking me.

Run cells with `Shift+Enter`. If VSCode asks for a kernel, pick the one in `.venv`.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # hush TensorFlow's startup noise

import tensorflow as tf
print(tf.__version__)

2.21.0


---
# Round 1 — Shape

## Learn

A tensor is a grid of numbers. Its **shape** tells you how big the grid is.

In [2]:
x = tf.constant([[1., 2., 3., 4.],
                 [5., 6., 7., 8.]])

print(x.numpy())
print("shape:", x.shape)

[[1. 2. 3. 4.]
 [5. 6. 7. 8.]]
shape: (2, 4)


`(2, 4)` means **2 rows, 4 columns**. The two directions are numbered:

- **axis 0** = down the rows
- **axis 1** = across the columns

In this project: **rows are paths, columns are time.**

## Practice 1

Print the shape of `y`. Say the answer out loud *before* you run it.

In [3]:
y = tf.constant([[1., 2., 3., 4., 5.],
                 [6., 7., 8., 9., 10.]])

print(y.numpy)
print("SHape:", y.shape)


# expect: (2, 5)

<bound method _EagerTensorBase.numpy of <tf.Tensor: shape=(2, 5), dtype=float32, numpy=
array([[ 1.,  2.,  3.,  4.,  5.],
       [ 6.,  7.,  8.,  9., 10.]], dtype=float32)>>
SHape: (2, 5)


---
# Round 2 — Dropping the first column

## Learn

Slicing syntax is `x[rows, columns]`.

- `:` means **all of them**
- `1:` means **from index 1 onwards** — i.e. drop the first

In [4]:
print("x was:")
print(x.numpy())
print()
print("x[:, 1:]  =")
print(x[:, 1:].numpy())
print("shape:", x[:, 1:].shape)

x was:
[[1. 2. 3. 4.]
 [5. 6. 7. 8.]]

x[:, 1:]  =
[[2. 3. 4.]
 [6. 7. 8.]]
shape: (2, 3)


## Practice 2

Do the same to `y`.

In [ ]:
# TODO: print y with its first column dropped, and the resulting shape


# expect values: [[ 2.  3.  4.  5.]
#                 [ 7.  8.  9. 10.]]
# expect shape:  (2, 4)

tf.Tensor(
[[ 2.  3.  4.  5.]
 [ 7.  8.  9. 10.]], shape=(2, 4), dtype=float32)


---
# Round 3 — Dropping the last column

## Learn

`:-1` means **up to but not including the last** — i.e. drop the last.

In [ ]:
print("x[:, :-1] =")
print(x[:, :-1].numpy())
print("shape:", x[:, :-1].shape)

## Practice 3

Do the same to `y`.

In [7]:
print (y[:,:-1].shape)

(2, 4)


---
# Round 4 — Subtracting the two slices

## Learn

Both slices have the **same shape**, and they are the same list shifted by one:

```
drop last   1   2   3        <- the "before" value
drop first      2   3   4    <- the "after" value
```

Subtract them and you get **the change from each value to the next**.

With prices, that change is the price move.

In [ ]:
prices = tf.constant([[100., 110., 105., 120.]])

moves = prices[:, 1:] - prices[:, :-1]

print("prices:", prices.numpy(), "shape", prices.shape)
print("moves: ", moves.numpy(),  "shape", moves.shape)

**4 prices give 3 moves.** 100→110 is +10, 110→105 is −5, 105→120 is +15.

This is exactly why `spot` has `n_steps + 1` columns but `delta` has only `n_steps`:
you make one decision per *gap*, not per price.

## Practice 4

Compute the moves for `p`.

In [9]:
p = tf.constant([[50., 55., 53.]])

p_moves = p[:,1:]-p[:,:-1]

print(p_moves)
# expect: [[ 5. -2.]]   shape (1, 2)

tf.Tensor([[ 5. -2.]], shape=(1, 2), dtype=float32)


---
# Round 5 — Adding up, and `axis`

## Learn

`tf.reduce_sum` adds numbers up. The `axis` argument says **which direction** to add in.

> **The rule: the axis you name is the axis that disappears.**

In [10]:
print(x.numpy(), "  shape", x.shape)
print()
print("axis=0  (rows disappear, add DOWN each column):")
print(" ", tf.reduce_sum(x, axis=0).numpy(), " shape", tf.reduce_sum(x, axis=0).shape)
print()
print("axis=1  (columns disappear, add ACROSS each row):")
print(" ", tf.reduce_sum(x, axis=1).numpy(), " shape", tf.reduce_sum(x, axis=1).shape)

[[1. 2. 3. 4.]
 [5. 6. 7. 8.]]   shape (2, 4)

axis=0  (rows disappear, add DOWN each column):
  [ 6.  8. 10. 12.]  shape (4,)

axis=1  (columns disappear, add ACROSS each row):
  [10. 26.]  shape (2,)


Started with `(2, 4)`:

| call | axis named | length of that axis | result shape |
|:--|:--|:--|:--|
| `axis=0` | 0 (rows) | 2 | `(4,)` |
| `axis=1` | 1 (columns) | 4 | `(2,)` |

**For us:** rows are paths, columns are time. We want *one number per path*, shape
`(n_paths,)`. So the **time** axis is the one that must disappear.

## Practice 5

`z` has shape `(2, 3)`. Write down both answers before running.

In [13]:
z = tf.constant([[1., 2., 3.],
                 [4., 5., 6.]])

print(tf.reduce_sum(z,axis=0))
print(tf.reduce_sum(z,axis=1))

tf.Tensor([5. 7. 9.], shape=(3,), dtype=float32)
tf.Tensor([ 6. 15.], shape=(2,), dtype=float32)


---
# Round 6 — `axis=-1`

## Learn

`-1` means **"the last axis"**, counting backwards. For a 2-D grid, `axis=-1` is the same
as `axis=1`.

In [14]:
print(tf.reduce_sum(x, axis=1).numpy())
print(tf.reduce_sum(x, axis=-1).numpy())

[10. 26.]
[10. 26.]


**Use `axis=-1` as your habit.** Later an extra dimension may appear at the front; `-1`
keeps meaning "time", while `1` would silently start meaning something else.

## Practice 6

Rewrite your second answer from Practice 5 using `axis=-1`. Same result.

In [20]:
print(tf.reduce_sum(z,axis=-1))

tf.Tensor([ 6. 15.], shape=(2,), dtype=float32)


---
# Round 7 — Multiplying two grids

## Learn

`*` between two grids of the **same shape** multiplies position by position.
It is *not* matrix multiplication.

In [ ]:
a = tf.constant([[1., 2., 3.]])
b = tf.constant([[10., 20., 30.]])

print(a.numpy(), "*", b.numpy(), "=", (a * b).numpy())

**A trap worth seeing now.** If the shapes *don't* match, TensorFlow often does **not**
raise an error — it stretches one side to fit and hands you a plausible wrong answer.

In [ ]:
print((tf.constant([[1., 2., 3.]]) * tf.constant([2.])).numpy())   # no error!

So: **print `.shape` constantly.** "It didn't crash" is not evidence that it is right.

## Practice 7

Multiply the position held by the price move, for each gap.

In [ ]:
held  = tf.constant([[1., 2.]])     # position held over each gap
gaps  = tf.constant([[5., -2.]])    # price move over each gap

gain_per_gap = None   # TODO

print(gain_per_gap)
# expect: [[ 5. -4.]]   shape (1, 2)

---
# Round 8 — Put it together: `hedging_gains`

## Learn

The formula from `docs/00-problem-statement.md`:

$$\text{gains} = \sum_{i=0}^{n-1} \delta_i \,(S_{i+1} - S_i)$$

In words: **for each gap in time, multiply the position you were holding by the price move
over that gap; add them up, separately for each path.**

That is Round 4 (moves) + Round 7 (multiply) + Round 6 (sum over time). You already have
all three.

## Practice 8 — one path

Check it by hand first:

| gap | position held | price move | gain |
|:--|:--|:--|:--|
| 100 → 110 | 1 | +10 | +10 |
| 110 → 105 | 2 | −5 | −10 |
| 105 → 120 | −1 | +15 | −15 |

**Total = −15.**

In [ ]:
spot  = tf.constant([[100., 110., 105., 120.]])   # 1 path, 4 prices
delta = tf.constant([[1.,   2.,  -1.]])           # 1 path, 3 positions

gains = None   # TODO: moves, then multiply, then sum over time

print(gains)
# expect: [-15.]   shape (1,)

## Practice 9 — two paths

One path can hide an `axis` mistake, because `(1,)` and `(3,)` both let you index `[0]`.
Run the **same code** on two paths.

If you get shape `(3,)` instead of `(2,)`, you summed the wrong axis — go back to Round 5.

In [ ]:
spot2  = tf.constant([[100., 110., 105., 120.],
                      [100.,  90.,  95.,  85.]])
delta2 = tf.constant([[1., 2., -1.],
                      [0., 1.,  1.]])

gains2 = None   # TODO: same code, using spot2 and delta2

print(gains2)
# expect: [-15. -5.]   shape (2,)

---
# Round 9 — Move it into the real file

Once Practice 9 prints `[-15. -5.]`, open `dhbench/pnl.py`, find `hedging_gains`, and
replace `raise NotImplementedError` with the code you just wrote — using the argument
names `spot` and `delta`.

Then run the three tests that don't need the (not yet written) GBM simulator:

```
python -m pytest tests/test_rung2_pnl_accounting.py -k hedging_gains -v
```

Three passes. Then come back and tell me — `transaction_costs` is next, and that one has
the interesting bug in it.